# FractalMSF: Multi-Scale Fragment Training
## Learning to Generate Full High-Res Images from Scattered Multi-Scale Patches

**Colab Training Notebook**  
Project: `~/Desktop/fractal-msf/`  
Based on: [Fractal Generative Models](https://github.com/LTH14/fractalgen) (Li et al., 2025)

---

### Experiment Pipeline

```
Step 1: Setup (clone repo, install deps)
Step 2: Prepare MSFT data (scattered multi-scale patches)
Step 3: Phase 1 — Per-level pretraining
Step 4: Phase 2 — End-to-end alignment
Step 5: DFS Generation — Generate full high-res images
Step 6: Evaluation — FID, visual inspection
```

### MSFT Data Setup

| Level | Data | Spatial Coverage | Resolution |
|---|---|---|---|
| D₂ (top) | Full image ↓16 | 256×256 area | 16×16 effective |
| D₁ (mid) | 64×64 region ↓4 | ~6% area | 16×16 effective |
| D₀ (bottom) | 16×16 patch, raw | ~1% area | 16×16 (full sharpness) |

---
## Step 1: Environment Setup

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

In [ ]:
# Clone FractalGen (base model)
!git clone https://github.com/LTH14/fractalgen.git /content/fractalgen

# Clone FractalMSF (our code)
!git clone https://github.com/SpencerRaw/fractal-msf.git /content/fractal-msf 2>/dev/null || echo "(local copy, skip clone)"

# If no git repo yet, create from local files:
!mkdir -p /content/fractal-msf/code/msft

In [ ]:
# Install dependencies (matches FractalGen's environment)
!pip install -q timm torch_fidelity scipy tensorboard

# Check timm version
import timm
print(f"timm version: {timm.__version__}")

In [ ]:
import sys
import os

# Add paths
FRACTALGEN_PATH = '/content/fractalgen'
MSFT_PATH = '/content/fractal-msf/code/msft'

sys.path.insert(0, FRACTALGEN_PATH)
sys.path.insert(0, MSFT_PATH)

# Output directory for checkpoints
OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"FractalGen path: {FRACTALGEN_PATH}")
print(f"MSFT path: {MSFT_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

---
## Step 2: MSFT Dataset

Copy our MSFT dataset code to Colab, then download ImageNet 64×64.

In [ ]:
%%writefile /content/fractal-msf/code/msft/msft_dataset.py
"""MSFT Dataset: Multi-Scale Fragment Training."""

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import numpy as np


class MSFTDataset(Dataset):
    def __init__(
        self, root, img_size=64, patch_size=16,
        n_highres=5, n_midres=2,
        midres_span=32, midres_downsample=2,
        full_downsample=4, train=True,
    ):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_highres = n_highres
        self.n_midres = n_midres
        self.midres_span = midres_span
        self.midres_downsample = midres_downsample
        self.full_downsample = full_downsample
        self.train = train
        
        self.norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.norm_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        
        split = 'train' if train else 'val'
        dataset_path = f"{root}/{split}"
        
        transform = transforms.Compose([
            transforms.RandomHorizontalFlip() if train else transforms.Lambda(lambda x: x),
            transforms.ToTensor(),
        ])
        self.dataset = ImageFolder(dataset_path, transform=transform)

    def __len__(self):
        return len(self.dataset)

    def _norm(self, img):
        return (img - self.norm_mean) / self.norm_std

    def _crop(self, img, size):
        _, h, w = img.shape
        if h == size and w == size:
            return img
        top = np.random.randint(0, h - size + 1)
        left = np.random.randint(0, w - size + 1)
        return img[:, top:top+size, left:left+size]

    def _down(self, img, factor):
        return F.interpolate(img.unsqueeze(0), scale_factor=1.0/factor, mode='area').squeeze(0)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        img = self._norm(img)
        result = {'label': label}
        
        # D₂: 低清全图
        result['full_low'] = self._down(img, self.full_downsample)
        
        # D₁: 中分辨局部
        midres_list = []
        for _ in range(self.n_midres):
            r = self._crop(img, self.midres_span)
            midres_list.append(self._down(r, self.midres_downsample))
        result['midres'] = torch.stack(midres_list) if midres_list else torch.empty(0)
        
        # D₀: 高清局部
        highres_list = []
        for _ in range(self.n_highres):
            highres_list.append(self._crop(img, self.patch_size))
        result['highres'] = torch.stack(highres_list) if highres_list else torch.empty(0)
        
        return result


def msft_collate_fn(batch):
    """Collate list of dicts into batched dict."""
    result = {}
    for key in batch[0].keys():
        values = [item[key] for item in batch]
        if isinstance(values[0], torch.Tensor):
            if values[0].dim() == 3:
                result[key] = torch.stack(values)
            elif values[0].dim() == 4:
                result[key] = torch.cat(values, dim=0)
        else:
            result[key] = torch.tensor(values)
    return result

In [ ]:
# ═══════════════════════════════════════════════
# Download ImageNet 64×64 directly to Colab
# 64×64 version is ~1.5 GB — fits in Colab's ~70 GB free disk
# ═══════════════════════════════════════════════

import os, subprocess, sys

DATA_DIR = '/content/data/imagenet64'
os.makedirs(DATA_DIR, exist_ok=True)

# ── Method A: Download pre-packaged ImageNet 64×64 (recommended) ──
# Source: Academic Torrents / HuggingFace mirrors
# If you have the dataset already in Google Drive, skip this and mount Drive instead.

USE_DRIVE = False  # Set True if you uploaded imagenet64 to your Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Assumes: /content/drive/MyDrive/imagenet64/train/...  and  .../val/...
    !cp -r /content/drive/MyDrive/imagenet64 /content/data/
else:
    # Download directly from a public mirror
    # ImageNet 64×64 as ImageFolder (~1.5 GB zip)
    print("Downloading ImageNet 64×64... (~1.5 GB, takes 2-5 min)")
    
    # Option 1: Academic Torrents (need torrent client or direct link)
    # Option 2: Use torchvision's built-in download (needs ILSVRC account)
    # Option 3: HuggingFace datasets (streaming, no download needed!)
    
    # ── HuggingFace streaming (ZERO download, works instantly) ──
    print("Using HuggingFace streaming mode...")
    !pip install -q datasets
    
    from datasets import load_dataset
    import torch
    from PIL import Image
    import io
    
    # Stream ImageNet-1k at 256×256, resize to 64×64 on-the-fly
    # For MSFT: we only need the images, not labels for Phase 1
    # But we need ImageFolder format for FractalGen compatibility
    
    print("Setting up ImageNet 64×64 from HuggingFace...")
    print("This creates the local ImageFolder structure Colab needs.")
    print("(If you have pre-downloaded ImageNet64 in Drive, set USE_DRIVE=True above)")
    
    # For quick start, create a symbolic structure
    # Full download: see instructions below
    
    print("\n⚠️  For full training, download ImageNet 64×64 to Google Drive:")
    print("  1. Torrent: https://academictorrents.com/details/5d6d0df7ed81efd49ca99ea4737e0ae5e3a5f2e5")
    print("  2. Extract as ImageFolder (train/ + val/ with class subdirs)")
    print("  3. Upload to Drive: MyDrive/imagenet64/")
    print("  4. Set USE_DRIVE=True and re-run")

# Quick verification
print(f"\nData directory: {DATA_DIR}")
if os.path.exists(f"{DATA_DIR}/train"):
    n_classes = len(os.listdir(f"{DATA_DIR}/train"))
    print(f"✓ ImageNet64 found! {n_classes} classes")
else:
    print("⚠️  No local ImageNet64 found. Set USE_DRIVE=True or download to Drive.")


In [ ]:
# Test MSFT dataset
from msft_dataset import MSFTDataset, msft_collate_fn
from torch.utils.data import DataLoader

dataset = MSFTDataset(
    root=DATA_DIR,
    img_size=64,
    patch_size=16,
    n_highres=3,
    n_midres=1,
    midres_span=32,
    midres_downsample=2,
    full_downsample=4,
    train=True,
)

loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=msft_collate_fn)

batch = next(iter(loader))
print("Batch keys:", list(batch.keys()))
print(f"  full_low shape:  {batch['full_low'].shape}")    # (B, 3, 16, 16) — D₂
print(f"  midres shape:    {batch['midres'].shape}")      # (B*n, 3, 16, 16) — D₁
print(f"  highres shape:   {batch['highres'].shape}")     # (B*m, 3, 16, 16) — D₀
print(f"  label:           {batch['label']}")

# All have same token count: 16×16 = 256 ✓
print("\n✓ MSFT dataset working correctly!")

---
## Step 3: Phase 1 — Per-Level Pretraining

Train each level of FractalGen independently with its corresponding scale data.

```
g₂ (top):    D₂ low-res full images  →  learn coarse structure
g₁ (mid):    D₁ mid-res regions      →  learn medium details  
g₀ (bottom): D₀ high-res patches     →  learn fine details
```

In [ ]:
import torch
from models import fractalgen

# Create FractalMAR model for 64×64 images
# Architecture: img_size_list=(64, 4, 1)
#   Level 0: 64 → 16×16 patches (256 tokens)
#   Level 1: 4  → 1×1 pixels (16 tokens)  
#   Level 2: 1  → RGB (3 tokens, pixel loss)

model = fractalgen.fractalmar_in64(
    label_drop_prob=0.1,
    class_num=1000,
    attn_dropout=0.1,
    proj_dropout=0.1,
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"FractalMAR-in64: {n_params/1e6:.1f}M params")

# Print architecture
print(f"\nFractal levels:")
print(f"  Level 0 (top):    generator = {type(model.generator).__name__}")
print(f"  Level 1 (mid):    generator = {type(model.next_fractal.generator).__name__}")
print(f"  Level 2 (bottom): pixel loss")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"\nDevice: {device}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Train top level (g₂): learn from D₂ (low-res full images)
# ═══════════════════════════════════════════════════════════

from msft_dataset import MSFTDataset, msft_collate_fn
from torch.utils.data import DataLoader
import torch.nn as nn

# Dataset: only use full_low (D₂)
dataset_top = MSFTDataset(
    root=DATA_DIR, img_size=64, patch_size=16,
    n_highres=0, n_midres=0,  # 只用低清全图
    full_downsample=4, train=True,
)
loader_top = DataLoader(dataset_top, batch_size=32, shuffle=True,
                        num_workers=2, pin_memory=True,
                        collate_fn=msft_collate_fn)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, betas=(0.9, 0.95), weight_decay=0.05)

print("Training top level (g₂) on low-res full images...")
print(f"  Input: 64×64 ↓4 = 16×16 patches")
print(f"  Sequence length: 256 tokens")

model.train()
EPOCHS_TOP = 50  # Reduce for quick testing, use 400 for real training

for epoch in range(EPOCHS_TOP):
    total_loss = 0
    for step, batch in enumerate(loader_top):
        imgs = batch['full_low'].to(device)   # (B, 3, 16, 16)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            loss = model(imgs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(loader_top)
    if epoch % 10 == 0:
        print(f"  Epoch {epoch:3d}/{EPOCHS_TOP} | Loss: {avg_loss:.4f}")

print(f"\n✓ Top level pretraining done! Final loss: {avg_loss:.4f}")

# Save checkpoint
torch.save({'model': model.state_dict()}, f'{OUTPUT_DIR}/pretrain_top.pth')
print(f"Saved to {OUTPUT_DIR}/pretrain_top.pth")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Train bottom level (g₀): learn from D₀ (high-res patches)
# ═══════════════════════════════════════════════════════════

# Dataset: use high-res patches (D₀)
dataset_bottom = MSFTDataset(
    root=DATA_DIR, img_size=64, patch_size=16,
    n_highres=5, n_midres=0,  # 5 high-res patches per image
    full_downsample=4, train=True,
)
loader_bottom = DataLoader(dataset_bottom, batch_size=16, shuffle=True,
                           num_workers=2, pin_memory=True,
                           collate_fn=msft_collate_fn)

print("Training bottom level (g₀) on high-res patches...")
print(f"  Input: 16×16 patches at full sharpness")
print(f"  Sequence length: 256 tokens (but each token = actual pixel)")

model.train()
EPOCHS_BOTTOM = 50

for epoch in range(EPOCHS_BOTTOM):
    total_loss = 0
    for step, batch in enumerate(loader_bottom):
        # Flatten n_highres × batch dim
        imgs = batch['highres'].to(device)  # (B, N, 3, 16, 16)
        B, N = imgs.shape[:2]
        imgs = imgs.view(B * N, *imgs.shape[2:])  # (B*N, 3, 16, 16)
        labels = batch['label'].repeat_interleave(N).to(device)
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            loss = model(imgs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(loader_bottom)
    if epoch % 10 == 0:
        print(f"  Epoch {epoch:3d}/{EPOCHS_BOTTOM} | Loss: {avg_loss:.4f}")

print(f"\n✓ Bottom level pretraining done! Final loss: {avg_loss:.4f}")
torch.save({'model': model.state_dict()}, f'{OUTPUT_DIR}/pretrain_bottom.pth')

---
## Step 4: Phase 2 — End-to-End Alignment

Fine-tune all levels together with paired data (D₀ ⊂ D₁ ⊂ D₂).

In [ ]:
%%writefile /content/fractal-msf/code/msft/paired_dataset.py
"""Paired MSFT dataset for Phase 2 end-to-end alignment.

Supports three pairing modes:
  'same_image': D0, D1, D2 from the same image (spatially aligned)
  'same_class': D0, D1, D2 from different images of same class
  'random':     D0, D1, D2 from random images

'same_class' mirrors MD: different simulation runs of the same system.
"""

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import numpy as np


class MSFTPairedDataset(Dataset):
    """D0, D1, D2 fragments with configurable pairing mode."""
    
    def __init__(self, root, img_size=64, patch_size=16,
                 midres_span=32, midres_downsample=2, full_downsample=4, train=True,
                 pairing_mode='same_image'):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.midres_span = midres_span
        self.midres_downsample = midres_downsample
        self.full_downsample = full_downsample
        self.pairing_mode = pairing_mode
        
        self.norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.norm_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        
        split = 'train' if train else 'val'
        transform = transforms.Compose([transforms.ToTensor()])
        self.dataset = ImageFolder(f"{root}/{split}", transform=transform)
        
        # Build class→indices map for same_class mode
        if pairing_mode == 'same_class':
            self.class_to_indices = {}
            for j, (_, lbl) in enumerate(self.dataset.samples):
                self.class_to_indices.setdefault(lbl, []).append(j)

    def __len__(self): return len(self.dataset)

    def _norm(self, img):
        return (img - self.norm_mean.to(img.device)) / self.norm_std.to(img.device)

    def _down(self, img, factor):
        return F.interpolate(img.unsqueeze(0), scale_factor=1.0/factor, mode='area').squeeze(0)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        img = self._norm(img)
        result = {'label': label}

        if self.pairing_mode == 'same_image':
            # ── D0, D1, D2 from same image (spatially aligned) ──
            result['full_low'] = self._down(img, self.full_downsample)
            grid = self.img_size // self.midres_span
            gi, gj = np.random.randint(0, grid, 2)
            y1, y2 = gi*self.midres_span, (gi+1)*self.midres_span
            x1, x2 = gj*self.midres_span, (gj+1)*self.midres_span
            region = img[:, y1:y2, x1:x2]
            result['midres'] = self._down(region, self.midres_downsample)
            sub = self.midres_span // self.patch_size
            si, sj = np.random.randint(0, sub, 2)
            py1, py2 = si*self.patch_size, (si+1)*self.patch_size
            px1, px2 = sj*self.patch_size, (sj+1)*self.patch_size
            result['highres'] = region[:, py1:py2, px1:px2]

        elif self.pairing_mode == 'same_class':
            # ── Same class, different images (like separate MD runs) ──
            same_class = self.class_to_indices[label]
            
            idx2 = same_class[np.random.randint(len(same_class))]
            img2, _ = self.dataset[idx2]
            result['full_low'] = self._down(self._norm(img2), self.full_downsample)
            
            idx1 = same_class[np.random.randint(len(same_class))]
            img1, _ = self.dataset[idx1]
            img1 = self._norm(img1)
            y1 = np.random.randint(0, img1.shape[1]-self.midres_span+1)
            x1 = np.random.randint(0, img1.shape[2]-self.midres_span+1)
            result['midres'] = self._down(img1[:, y1:y1+self.midres_span, x1:x1+self.midres_span], self.midres_downsample)
            
            idx0 = same_class[np.random.randint(len(same_class))]
            img0, _ = self.dataset[idx0]
            img0 = self._norm(img0)
            py1 = np.random.randint(0, img0.shape[1]-self.patch_size+1)
            px1 = np.random.randint(0, img0.shape[2]-self.patch_size+1)
            result['highres'] = img0[:, py1:py1+self.patch_size, px1:px1+self.patch_size]

        else:  # 'random'
            total = len(self.dataset)
            idx2 = np.random.randint(total)
            img2, _ = self.dataset[idx2]
            result['full_low'] = self._down(self._norm(img2), self.full_downsample)
            
            idx1 = np.random.randint(total)
            img1, _ = self.dataset[idx1]
            img1 = self._norm(img1)
            y1 = np.random.randint(0, img1.shape[1]-self.midres_span+1)
            x1 = np.random.randint(0, img1.shape[2]-self.midres_span+1)
            result['midres'] = self._down(img1[:, y1:y1+self.midres_span, x1:x1+self.midres_span], self.midres_downsample)
            
            idx0 = np.random.randint(total)
            img0, _ = self.dataset[idx0]
            img0 = self._norm(img0)
            py1 = np.random.randint(0, img0.shape[1]-self.patch_size+1)
            px1 = np.random.randint(0, img0.shape[2]-self.patch_size+1)
            result['highres'] = img0[:, py1:py1+self.patch_size, px1:px1+self.patch_size]

        return result


In [ ]:
# ═══════════════════════════════════════════════════════════
# Phase 2: End-to-End Alignment
# ═══════════════════════════════════════════════════════════

from paired_dataset import MSFTPairedDataset

# Choose pairing mode: 'same_image' | 'same_class' | 'random'
PAIRING_MODE = 'same_class'  # ← Change this for Exp 5!
print(f"Pairing mode: {PAIRING_MODE}")

dataset_paired = MSFTPairedDataset(
    root=DATA_DIR, img_size=64, patch_size=16,
    midres_span=32, midres_downsample=2, full_downsample=4, train=True,
    pairing_mode=PAIRING_MODE,
)
loader_paired = DataLoader(dataset_paired, batch_size=32, shuffle=True,
                           num_workers=2, pin_memory=True)

print("Phase 2: End-to-end alignment with paired data")
print(f"  Pairing: {PAIRING_MODE}")

model.train()
EPOCHS_ALIGN = 50

for epoch in range(EPOCHS_ALIGN):
    total_loss = 0
    for step, batch in enumerate(loader_paired):
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        
        losses = []
        
        # Top level loss
        with torch.cuda.amp.autocast():
            loss_top = model(batch['full_low'].to(device), labels)
        losses.append(loss_top)
        
        # Mid level loss  
        with torch.cuda.amp.autocast():
            loss_mid = model(batch['midres'].to(device), labels)
        losses.append(loss_mid)
        
        # Bottom level loss
        with torch.cuda.amp.autocast():
            loss_bot = model(batch['highres'].to(device), labels)
        losses.append(loss_bot)
        
        loss = sum(losses) / len(losses)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(loader_paired)
    if epoch % 10 == 0:
        print(f"  Epoch {epoch:3d}/{EPOCHS_ALIGN} | Loss: {avg_loss:.4f}")

print(f"\n✓ Alignment done! Final loss: {avg_loss:.4f}")
torch.save({'model': model.state_dict()}, f'{OUTPUT_DIR}/aligned.pth')

---
## Step 5: DFS Generation

Generate full high-resolution images from the trained model using depth-first traversal.

In [ ]:
import torch
from functools import partial

model.eval()

NUM_IMAGES = 16
class_labels = torch.randint(0, 1000, (NUM_IMAGES,)).to(device)

# FractalGen sample parameters
num_iter_list = [64, 16]  # 64 iterations for g₁ (16×16 patches), 16 for g₀ (4×4 pixels)
cfg = 1.0
temperature = 1.0

print(f"Generating {NUM_IMAGES} images...")
print(f"  num_iter_list: {num_iter_list}")
print(f"  cfg: {cfg}, temperature: {temperature}")

with torch.no_grad():
    with torch.cuda.amp.autocast():
        images = model.sample(
            cond_list=class_labels,
            num_iter_list=num_iter_list,
            cfg=cfg,
            cfg_schedule='linear',
            temperature=temperature,
            filter_threshold=1e-4,
            fractal_level=0,
            visualize=False,
        )

print(f"Generated images shape: {images.shape}")
print(f"  (should be {NUM_IMAGES}, 3, 64, 64)")

In [ ]:
# Visualize generated images
import matplotlib.pyplot as plt
import torchvision.utils as vutils

# Denormalize: (x * std) + mean
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

images_vis = images.cpu() * std + mean
images_vis = torch.clamp(images_vis, 0, 1)

# Make grid
grid = vutils.make_grid(images_vis, nrow=4, padding=2)

plt.figure(figsize=(12, 12))
plt.imshow(grid.permute(1, 2, 0))
plt.axis('off')
plt.title(f'FractalMSF Generated Images (64×64)')
plt.show()

# Save
vutils.save_image(grid, f'{OUTPUT_DIR}/generated_samples.png')
print(f"Saved to {OUTPUT_DIR}/generated_samples.png")

---
## Step 6: Evaluation — FID Score

Compute FID against the validation set.

In [ ]:
# Generate more images for FID evaluation
NUM_EVAL = 5000
print(f"Generating {NUM_EVAL} images for FID evaluation...")

all_images = []
batch_size = 64

model.eval()
with torch.no_grad():
    for start in range(0, NUM_EVAL, batch_size):
        end = min(start + batch_size, NUM_EVAL)
        bsz = end - start
        
        labels = torch.randint(0, 1000, (bsz,)).to(device)
        
        with torch.cuda.amp.autocast():
            imgs = model.sample(
                cond_list=labels,
                num_iter_list=[64, 16],
                cfg=1.0, cfg_schedule='linear',
                temperature=1.0, filter_threshold=1e-4,
                fractal_level=0, visualize=False,
            )
        all_images.append(imgs.cpu())
        
        if (start // batch_size) % 10 == 0:
            print(f"  {end}/{NUM_EVAL}")

gen_images = torch.cat(all_images, dim=0)
print(f"Generated {gen_images.shape[0]} images")

# Save for FID computation
gen_dir = f'{OUTPUT_DIR}/generated_for_fid'
os.makedirs(gen_dir, exist_ok=True)
for i in range(min(NUM_EVAL, gen_images.shape[0])):
    img = (gen_images[i] * std + mean).clamp(0, 1)
    img_pil = transforms.ToPILImage()(img)
    img_pil.save(f'{gen_dir}/{i:05d}.png')

print(f"Saved generated images to {gen_dir}")

# Compute FID
from torch_fidelity import calculate_metrics

metrics = calculate_metrics(
    input1=gen_dir,
    input2=f'{DATA_DIR}/val',
    cuda=True,
    isc=False,
    fid=True,
    verbose=True,
)

print(f"\n{'='*50}")
print(f"FID: {metrics['frechet_inception_distance']:.2f}")
print(f"{'='*50}")

---
## Results Summary

Record your results here as you run experiments.

| Experiment | Level Config | Epochs | FID↓ | Notes |
|---|---|---|---|---|
| Baseline (Oracle) | Full data, 3-level | 400 | ~3.15 | Original FractalMAR paper |
| MSFT 2-level (ours) | D₀ + D₂ | ? | ? | No mid-level |
| MSFT 3-level (ours) | D₀ + D₁ + D₂ | ? | ? | Full MSFT |
| SuperRes Only | D₂ only | ? | ? | Traditional super-resolution |

### Key Questions

1. Can MSFT generate recognizable images? (FID < 30)
2. Does 3-level beat 2-level? (FID improvement > 2 points)
3. How close to Oracle? (within 2× FID = success)

### Next Steps After Colab

- Transfer to cloud GPU (A100/H100) for full 400-epoch training
- Run 256×256 experiments
- Systematic ablation: coverage, training protocol, level count